<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #1e3c72; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Découverte de données satellitaires - Productions Météo-France - Format TIFF</div>

---

## 🎯 **Objectifs du TP**

##  🔎 Analyser une image satellitaire TIFF géoréférencée
##  🔄 Visualiser une image

---

## 🧰 **Workflow : lignes de commandes bash (GDAL & ImageMagick)**

Ce TP utilise principalement la bibliothèque **GDAL** (*Geospatial Data Abstraction Library*), un outil puissant pour manipuler les données issues de satellites météorologiques.

Les fichiers utilisés sont au format **TIFF**, issus de la production opérationnelle de Météo-France.


### Qu'est-ce qu'ImageMagick ?
ImageMagick est une suite **logicielle libre et open source** dédiée à la **création**, la **conversion**, la **modification** et l'**analyse** d'**images raster**. C'est un outil très utilisé en automatisation, traitement d'images scientifiques, développement web et production graphique. </br>
Il permet de manipuler plus de 200 formats d'images, notamment : JPEG, PNG, TIFF, GIF, PDF, SVG... </br>
Il fonctionne principalement en ligne de commande, mais peut aussi être intégré dans des scripts ou des applications via des bibliothèques disponibles en C, C++, Python, PHP, Java, etc. </br>
https://imagemagick.org</br>

---

## 📂 **Fichier utilisé**

Le fichier TIFF étudié est un produit de type **true color** du satellite Meteosat-12 :

📁 `/stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT`

> 💡 *Ce fichier est un produit opérationnel du Centre de Météorologie Spatiale (CMS) de Météo-France.*

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

Le premier fichier TIF proposé est le suivant : EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT </br>
Il se trouve dans ce répertoire : **`/stockage/DATA/TIF/20260422`**</br></br>
Il s'agit d'un fichier type d'une production CMS envoyée par Transmet (extension .LT).

In [ ]:
#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
INPUT="/stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT"
#INPUT="/stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT"
print(f"📁 Fichier source configuré : {INPUT}")

<div class="alert alert-info alert-success">
<h3> 1 - 🔎 Visualisation du TIFF et de son contenu</h3>
</div>

La commande `gdalinfo` affiche les métadonnées géospatiales d’un fichier raster. Dans Jupyter, on la précède de `!`.

In [ ]:
!gdalinfo $INPUT

## 📋 **Décryptage des informations clés**

| Information | Valeur | Signification |
|-------------|--------|----------------|
| 🏷️ **Driver** | GTiff/GeoTIFF | Format standard géoréférencé |
| 📛 **Fichier** | `EOXF71MTGG221200...` | Produit Meteosat true color, date 2026-04-22 12:00 UTC |
| 📐 **Taille** | 22272 × 22272 pixels | Image très haute résolution (~500 m/pixel) |
| 🛰️ **Projection** | Geostationary Satellite (Sweep Y *) | Vue depuis l'espace, satellite Meteosat |
| 📍 **Position** | Longitude 0°, altitude 35 786 km | Au-dessus de Greenwich |
| 🧭 **Origine** | (-5 568 000, 5 568 000) mètres | Coin haut gauche de l'image |
| 📏 **Pixel Size** | (500, -500) mètres | Résolution de 500 mètres par pixel |
| 🏷️ **TiffTAG** * | process:geostationary_meteosat_0_deg | aggregate:geostationary_true_color |
| 🕒 **Date image** | 2026-04-22 12:00:00 UTC | Heure du slot |
| 📏 **Espace colorimétrique** | YCbCr (compression JPEG) | Compression avec perte (JPEG) Qualité élevée |
| 🎨 **Bandes** | Rouge, Vert, Bleu | 8 bits par canal (valeurs 0-255) |
| 🗂️ **Pyramides** * | 11136, 5568, 2784, 1392 | Affichage rapide à différentes échelles ("Overviews" |

> 💡*Sweep Y : type de balayage utilisé par Meteosat (Miroir oscille selon l'axe Y (nord-sud).Pour information, satellites Goes américain : Sweep X*</br>
  *Les pyramides (overviews) sont nécessaire pour Synopsis : versions réduites de l'image intégrées au fichier pour un affichage rapide sans recalcul.*</br>
>  *Le TiffTAG est nécessaire pour Synopsis* 

In [ ]:
cd ~/MF_DATA_MANIPULATION

📁 **Création du dossier de résultats**

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'

## 🖼️ **Redimensionnement et conversion JPG**

Nous allons redimensionner l'image et la convertir en JPG. L'utilisation de l'overview `[3]` (image 2784×2784 pixels) permet un gain de temps.

In [ ]:
!convert -quiet -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg

### 🖼️ **Visualisation de l'image**

In [ ]:
im = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg', 'r')
display(im)

📌 **Description** : Image **Geocolor RGB** en vue géostationnaire (GEOS), depuis le satellite **Meteosat-12** positionné à 0° de longitude. Ce produit est réalisé à partir de fichiers multicanaux, avec diverses corrections (Diffusion, parallaxe, correction angle solaire).

> 💡 *Le produit Geocolor RGB combine plusieurs canaux pour restituer une image en "vraies couleurs" jour et nuit.*

<div class="alert alert-info alert-success">
<h3> 2 - ✂️ Découpes (cropping)</h3>
</div>

### 🇫🇷 **Découpe sur la France (projection EPSG:4326)**

L'avantage de la projection WGS84 est de pouvoir spécifier les limites en degrés.

**Options importantes** :
- `-t_srs {proj}` : nouvelle projection
- `-te {west} {south} {east} {north}` : étendue de découpe <strong>(W S E N)</strong>
- `-ts {width} {height}` : taille de sortie en pixels
- `-tr {xres} {yres}` : résolution spatiale

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 -te -14 38 18 56 -ts 960 540 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.tif
!convert $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg

In [ ]:
im2 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg', 'r')
display(im2)

> ⚠️ **Attention** : L'image peut être déformée si le ratio de sortie (`-ts`) ne correspond pas au ratio géographique de la zone découpée.</br>
> Ici, la taille 960x540 correspond a du 16/9.

<div class="alert alert-info alert-success">
<h3> 3 - 🖌️ Ajouter un contour</h3>
</div>

### 🗺️ **Ajout de contours avec `gdal_rasterize`**

La commande `gdal_rasterize` permet de dessiner des contours (fichiers shapefile) sur une image.

**Options de couleur** :
- `-b 1 -burn 255 -b 2 -burn 255 -b 3 -burn 255` → blanc
- `-b 1 -burn 0 -b 2 -burn 0 -b 3 -burn 0` → noir
- `-b 1 -burn 255 -b 2 -burn 0 -b 3 -burn 0` → rouge

> ⚠️ *Attention : `gdal_rasterize` modifie directement le fichier d'entrée (le TIF est écrasé).*

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -ts 960 540 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif
!gdal_rasterize -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp  $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif >/dev/null 2>&1
!convert -quiet -resize 1920x1080 $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg
im18 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg', 'r')
display(im18)

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

À partir d'un des autres tiffs disponibles pour cette journée :</br>
- Visualiser ce tiff : convertir en un jpg redimensionné en 1000x1000 </br>
- Découper ce tiff   : choisir le domaine, ajouter les frontières et convertir en jpg redimensionné </br>


Liste des tiffs plein disque disponibles pour le 22 avril 2026 à 12 h 00 UTC :</br> IR105, VIS006, Ash RGB ("cendre"), Dust RGB, General RGB (ccgene), Sandwich, Fire Temperature RGB (tempfeux), Cloud Type RGB, Cloupd Phase RGB

In [ ]:
!ls -ltr /stockage/DATA/TIF/20260422/globe*GTP*

## 🌟 **Ressources complémentaires**

| Lien | Description |
|------|-------------|
| [Galerie CMS](http://e-sat.cms.meteo.fr/GALERIE/archive.html) | Archives d'images satellitaires |
| [Animations temps réel](http://e-sat.cms.meteo.fr/GALERIE/animations.html) | Animations Meteosat en direct |
| [Documentation GDAL](https://gdal.org/en/stable/) | Référence complète de GDAL |

---

✅ **Fin du TP**